# YOLO26s @ imgsz=832 — CBVD-5 + CVB (mix pour ajouter `walking`)

**But** : ajouter la classe `walking` (absente de CBVD-5) au modèle en fusionnant avec CVB (CSIRO), le seul dataset public qui la sépare de `standing`.

**Datasets utilisés** :
- **CBVD-5** — 5 classes barn/étable, 4122 frames pré-annotées (Kaggle).
- **CVB** — 11 classes pâturage extérieur, 1.15M annotations sur 225k frames (CSIRO S3).
- MmCows + COLO **skippés** (licence NC + effort de conversion vs gain marginal).

**Hardware** : RTX 4090 24 GB, Ubuntu 22.04.

**Chemins absolus utilisés partout** — pas de `os.chdir` pour éviter les dossiers imbriqués.

**Ordre d'exécution** : cellules dans l'ordre. Chaque section est indépendante et re-runnable.

## 0 — Environnement

In [ ]:
%pip install -q ultralytics kaggle awscli pyyaml opencv-python
# Chaîne d'export CoreML (versions verrouillées qui marchent, sinon BlobWriter cassé).
%pip install -q 'coremltools<8' 'torch<2.5' 'torchvision<0.20'

In [ ]:
import os, torch, pathlib
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
!df -h /workspace | tail -1

## 1 — Credentials

Upload `kaggle.json` dans `/workspace/` avant d'exécuter.

In [ ]:
!mkdir -p ~/.kaggle
!cp /workspace/kaggle.json ~/.kaggle/kaggle.json 2>/dev/null || echo 'Upload /workspace/kaggle.json puis re-run.'
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null
!kaggle --version

## 2 — Download CBVD-5 (Kaggle, ~2 GB)

In [ ]:
!mkdir -p /workspace/datasets && cd /workspace/datasets && \
    kaggle datasets download -d fandaoerji/cbvd-5cow-behavior-video-dataset --unzip
!ls /workspace/datasets/ | head

## 3 — Download CVB (CSIRO S3, ~15 GB de frames)

Récupère l'access key CSIRO sur https://data.csiro.au/collection/csiro:58916v1 (accepte les T&Cs).

In [ ]:
import os, subprocess, pathlib

# ⚠️ Remplace par tes propres clés CSIRO (temporaires, expirent au bout de quelques jours).
os.environ['AWS_ACCESS_KEY_ID']     = 'VORII654W7HVEPX9BI36'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'hCWdlnBR76TJIb6AIhOwv/UkUUMff5+YbbEtIz3M'

!aws configure set default.s3.max_concurrent_requests 20
!aws configure set default.s3.multipart_chunksize 64MB
!aws configure set default.s3.multipart_threshold 64MB

DEST = pathlib.Path('/workspace/cvb')
DEST.mkdir(exist_ok=True)

before = int(subprocess.check_output(['du', '-sb', str(DEST)]).split()[0]) if DEST.exists() and any(DEST.iterdir()) else 0
print(f'Début sync — taille actuelle : {before/1e9:.2f} GB')
print('(sync silencieux — surveille avec `watch -n 30 "du -sh /workspace/cvb"` dans un terminal séparé)')

!aws s3 sync \
    --endpoint-url https://s3.data.csiro.au \
    --only-show-errors \
    s3://dapprd/000058916v001/ \
    /workspace/cvb/

after = int(subprocess.check_output(['du', '-sb', str(DEST)]).split()[0])
n = sum(1 for _ in DEST.rglob('*') if _.is_file())
print(f'\n✓ Sync terminé : {after/1e9:.2f} GB, {n} fichiers')

## 4 — Taxonomie unifiée

Les labels CBVD-5 (5 classes) et CVB (11 classes) sont remappés vers la taxonomie canonique 7 classes.

In [ ]:
CANONICAL = ['standing', 'lying', 'eating', 'walking', 'running', 'drinking', 'other']

ALIASES = {
    # ─── CBVD-5 (labelmap exact) ─────────────────────────────────────
    'stand':          'standing',
    'lying_down':     'lying',        # 'lying down' → normalisé 'lying_down'
    'foraging':       'eating',
    'drinking_water': 'drinking',
    'rumination':     'other',        # état ; écrasé si co-occurrence avec posture
    # ─── CVB (labelmap exact behaviour_list.pbtx) ────────────────────
    'none':                None,       # placeholder → drop (aucun sens visuel)
    'grazing':             'eating',
    'walking':             'walking',  # ← LA classe qui manque à CBVD-5
    'ruminating_standing': 'standing', # posture prime sur l'action ruminante
    'ruminating_lying':    'lying',
    'resting_standing':    'standing',
    'resting_lying':       'lying',
    'drinking':            'drinking',
    'grooming':            'other',    # se lécher / se gratter = comportement distinct
    'other':               'other',
    'hidden':              None,       # vache occluse → drop (pas de signal visuel utile)
    'running':             'running',
}

def to_canonical_id(src_label: str):
    s = src_label.strip().lower().replace('-', '_').replace(' ', '_')
    # .get() renvoie None si aliased vers None (=drop) OU si absent, alors on
    # regarde si le nom est directement dans CANONICAL, sinon None.
    if s in ALIASES:
        tgt = ALIASES[s]
    else:
        tgt = s if s in CANONICAL else None
    return CANONICAL.index(tgt) if tgt in CANONICAL else None

print(f'✓ {len(ALIASES)} aliases, {len(CANONICAL)} classes canoniques')

## 5 — Setup dossier merged + helpers

In [ ]:
import pathlib, random, shutil

MERGED = pathlib.Path('/workspace/merged').resolve()
for split in ('train', 'val'):
    (MERGED / 'images' / split).mkdir(parents=True, exist_ok=True)
    (MERGED / 'labels' / split).mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.10
random.seed(0)

def add_sample(img_path: pathlib.Path, yolo_lines, unique_name: str = None):
    """Symlink image + write label into train or val bucket.
    unique_name : si fourni, remplace img_path.name — nécessaire pour CVB dont
    les fichiers img_00001.jpg sont dupliqués à travers 500+ vidéos.
    """
    fname = unique_name or img_path.name
    split = 'val' if random.random() < VAL_RATIO else 'train'
    dst_img = MERGED / 'images' / split / fname
    dst_lbl = MERGED / 'labels' / split / (pathlib.Path(fname).stem + '.txt')
    if not dst_img.exists():
        try:    dst_img.symlink_to(img_path)
        except OSError: shutil.copy(img_path, dst_img)
    dst_lbl.write_text('\n'.join(yolo_lines) + '\n')

print(f'✓ Dossier merged prêt : {MERGED}')

## 6 — Converter AVA unifié (CBVD-5 et CVB parlent le même format)

Gère les 2 layouts de frames :
- CBVD-5 : `labelframes/<video_id>_<ts:05d>.jpg` (fichiers à plat)
- CVB : `raw_frames/<video_id>/img_<ts:05d>.jpg` (un dossier par vidéo)

Quand plusieurs actions taggent le même bbox (ex: `stand + rumination`), on garde l'action prioritaire (posture > rumination).

In [ ]:
import cv2, pathlib, re

ACTION_PRIORITY = ['drinking', 'eating', 'walking', 'running', 'lying', 'standing', 'other']
def _prio(name): return ACTION_PRIORITY.index(name) if name in ACTION_PRIORITY else 99

def _load_labelmap(path: pathlib.Path):
    m, txt = {}, path.read_text()
    for line in txt.splitlines():
        line = line.strip()
        if ':' in line and not line.startswith(('#', 'label', 'name', 'label_id')):
            k, v = line.split(':', 1)
            try: m[int(k.strip())] = v.strip()
            except ValueError: pass
    if not m:
        for match in re.finditer(r'name:\s*"([^"]+)".*?label_id:\s*(\d+)', txt, re.DOTALL):
            m[int(match.group(2))] = match.group(1)
    return m

def convert_ava(tag, ann_dir, videos_dir, labelframes_dir=None):
    ann_dir = pathlib.Path(ann_dir)
    videos_dir = pathlib.Path(videos_dir)
    labelframes_dir = pathlib.Path(labelframes_dir) if labelframes_dir else None

    if not ann_dir.exists():
        print(f'[{tag}] {ann_dir} absent, skip'); return
    labelmap_path = next(iter(ann_dir.glob('labelmap*')), None) or next(iter(ann_dir.glob('*.pbtxt')), None)
    if not labelmap_path:
        print(f'[{tag}] pas de labelmap dans {ann_dir}, skip'); return
    labelmap = _load_labelmap(labelmap_path)
    print(f'[{tag}] labelmap = {labelmap}')

    total, skipped = 0, 0
    for csv_path in sorted(ann_dir.glob('ava_*.csv')):
        if 'excluded' in csv_path.name: continue
        frames = {}
        with open(csv_path) as f:
            for line in f:
                p = line.strip().split(',')
                if len(p) < 7: continue
                try:
                    vid, ts = p[0], float(p[1])
                    x1, y1, x2, y2 = map(float, p[2:6])
                    aid = int(p[6])
                except ValueError:
                    continue
                src = labelmap.get(aid)
                if not src: continue
                cid = to_canonical_id(src)
                if cid is None: continue
                bkey = (round(x1, 4), round(y1, 4), round(x2, 4), round(y2, 4))
                frames.setdefault((vid, ts), {}).setdefault(bkey, []).append(CANONICAL[cid])

        for (vid, ts), bboxes in frames.items():
            yolo_lines = []
            for (x1, y1, x2, y2), names in bboxes.items():
                primary = min(names, key=_prio)
                cid = CANONICAL.index(primary)
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                bw, bh = x2 - x1, y2 - y1
                yolo_lines.append(f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            if not yolo_lines: continue

            frame_path = None
            if labelframes_dir:
                for cand in [
                    labelframes_dir / f'{vid}_{int(ts):05d}.jpg',       # CBVD-5 flat
                    labelframes_dir / f'{vid}_{int(ts)}.jpg',
                    labelframes_dir / vid / f'img_{int(ts):05d}.jpg',   # CVB nested
                    labelframes_dir / vid / f'img_{int(ts)}.jpg',
                ]:
                    if cand.exists(): frame_path = cand; break
            if frame_path is None:
                video_file = videos_dir / f'{vid}.mp4'
                if not video_file.exists(): skipped += 1; continue
                cap = cv2.VideoCapture(str(video_file))
                cap.set(cv2.CAP_PROP_POS_MSEC, ts * 1000)
                ok, frame = cap.read(); cap.release()
                if not ok: skipped += 1; continue
                frame_path = pathlib.Path(f'/tmp/_frames_{tag}/{vid}_{int(ts*1000)}.jpg')
                frame_path.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(frame_path), frame)

            # Nom GLOBALEMENT unique : tag + video_id + timestamp. Sans ça,
            # les 500+ vidéos CVB écrasent leurs 'img_00001.jpg' entre elles.
            unique_name = f'{tag}_{vid}_{int(ts):05d}.jpg'
            add_sample(frame_path, yolo_lines, unique_name=unique_name)
            total += 1

    print(f'[{tag}] ✓ {total} frames converties ({skipped} sautées faute de frame source)')

## 7 — Extraire le labelmap CVB (pas de fichier fourni, faut le déduire du COCO JSON)

In [ ]:
import re, pathlib

# CVB expédie son labelmap d'actions dans behaviour_list.pbtx (typo CSIRO : .pbtx
# et pas .pbtxt). Les JSON COCO contiennent seulement la classe objet 'cow',
# pas les noms d'actions — ils sont dans le pbtx.
pbtx = pathlib.Path('/workspace/cvb/data/cvb_in_ava_format/behaviour_list.pbtx')
if not pbtx.exists():
    raise FileNotFoundError(f'{pbtx} introuvable — CVB pas encore téléchargé ?')

content = pbtx.read_text()
mapping = {int(m.group(2)): m.group(1)
           for m in re.finditer(r'name:\s*"([^"]+)".*?label_id:\s*(\d+)', content, re.DOTALL)}
if not mapping:
    for line in content.splitlines():
        if ':' in line:
            try:
                k, v = line.split(':', 1)
                mapping[int(k.strip())] = v.strip().strip('"')
            except (ValueError, IndexError): pass

# Écrit un labelmap.txt standard à côté des CSVs.
lmpath = pathlib.Path('/workspace/cvb/data/cvb_in_ava_format/labelmap.txt')
lmpath.write_text('\n'.join(f'{k}: {v}' for k, v in sorted(mapping.items())) + '\n')

print(f'CVB labelmap ({len(mapping)} classes) → {lmpath}\n')
real_missing = []
for k, v in sorted(mapping.items()):
    norm = v.strip().lower().replace('-', '_').replace(' ', '_')
    if norm in ALIASES and ALIASES[norm] is None:
        mark = '→ ⊘ drop volontaire'
    else:
        cid = to_canonical_id(v)
        if cid is None:
            mark = '→ ❌ NON MAPPÉ (à ajouter à ALIASES)'
            real_missing.append(v)
        else:
            mark = f'→ ✓ {CANONICAL[cid]}'
    print(f'  {k}: {v:25s} {mark}')

if real_missing:
    print(f'\n⚠️  À ajouter à ALIASES : {real_missing}')
else:
    print('\n✅ Tous les mappings OK — lance la cellule 8 pour la conversion CVB.')

## 8 — Convertir CBVD-5 et CVB vers YOLO

In [ ]:
convert_ava(
    tag='cbvd5',
    ann_dir='/workspace/datasets/annotations',
    videos_dir='/workspace/datasets/videos/videos',
    labelframes_dir='/workspace/datasets/labelframes/labelframes',
)

In [ ]:
convert_ava(
    tag='cvb',
    ann_dir='/workspace/cvb/data/cvb_in_ava_format',
    videos_dir='/workspace/cvb/data/videos',   # inexistant → fallback frames
    labelframes_dir='/workspace/cvb/data/raw_frames',
)

## 9 — data.yaml + histogramme des classes

C'est ici que tu vas voir `walking` apparaître comme classe non-vide (grâce à CVB).

In [ ]:
import yaml

yaml_body = {
    'path':  str(MERGED),
    'train': 'images/train',
    'val':   'images/val',
    'names': CANONICAL,
}
(MERGED / 'data.yaml').write_text(yaml.safe_dump(yaml_body, sort_keys=False))
print(f'✓ {MERGED / "data.yaml"}\n')

counts = [0] * len(CANONICAL)
for lbl in (MERGED / 'labels').rglob('*.txt'):
    for line in lbl.read_text().splitlines():
        try: counts[int(line.split()[0])] += 1
        except (ValueError, IndexError): pass

print(f'Total annotations : {sum(counts)}')
print(f'Total frames      : {sum(1 for _ in (MERGED / "images").rglob("*.jpg"))}\n')
for name, n in zip(CANONICAL, counts):
    bar = '█' * min(60, n // max(1, max(counts) // 60))
    print(f'  {name:10s} {n:>8d}  {bar}')

## 10 — Training YOLO26s @ imgsz=832

**Warm-start** (recommandé) depuis ton `best.pt` existant. Upload-le d'abord à `/workspace/weights/prev_best.pt`.

Bascule `WARM_START = False` pour repartir de `yolo26s.pt` COCO (fresh, 80 epochs standard).

In [ ]:
from ultralytics import YOLO

WARM_START   = True
PREV_WEIGHTS = '/workspace/weights/prev_best.pt'

if WARM_START and pathlib.Path(PREV_WEIGHTS).exists():
    starting = PREV_WEIGHTS; epochs = 40; lr0 = 0.001; freeze = 10
    print(f'Warm-start depuis {starting}')
else:
    if WARM_START:
        print(f'⚠️  {PREV_WEIGHTS} introuvable → fresh start.')
    starting = 'yolo26s.pt'; epochs = 80; lr0 = 0.01; freeze = 0
    print(f'Fresh start depuis {starting} (COCO pretrained)')

model = YOLO(starting)
model.train(
    data='/workspace/merged/data.yaml',
    imgsz=832,
    epochs=epochs,
    batch=-1,               # autotune 24 GB
    patience=15,
    lr0=lr0,
    freeze=freeze,
    cos_lr=True,
    close_mosaic=10,
    device=0,
    project='/workspace/runs',
    name='boeuf_yolo26s_832_mixed',
    save=True, save_period=5,
)

## 10bis — Rééquilibrage frame-level (à faire si mAP walking << 0.20 après Run A)

Downsample les frames dominées par `standing/lying/eating` pour ramener le ratio walking:eating à ~1:3. Priorise la conservation des frames contenant des classes minoritaires (walking/running/drinking).

In [ ]:
import shutil, pathlib, random, collections, yaml

MERGED = pathlib.Path('/workspace/merged')
BAK = pathlib.Path('/workspace/merged_unbalanced')

# Restaure l'état d'origine.
if not BAK.exists():
    shutil.move(str(MERGED), str(BAK))
if MERGED.exists():
    shutil.rmtree(MERGED)
for split in ('train', 'val'):
    (MERGED / 'images' / split).mkdir(parents=True, exist_ok=True)
    (MERGED / 'labels' / split).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['standing', 'lying', 'eating', 'walking', 'running', 'drinking', 'other']
MINORITY = {3, 4, 5, 6}   # walking, running, drinking, other → keep frames containing them

def frame_stats(lbl_path):
    """Retourne (classe_dominante, set_de_classes_présentes)."""
    counts = collections.Counter()
    for line in lbl_path.read_text().splitlines():
        try: counts[int(line.split()[0])] += 1
        except: pass
    if not counts: return None, set()
    return counts.most_common(1)[0][0], set(counts.keys())

# Quotas AGRESSIFS sur les classes majoritaires. Les frames contenant une classe
# minoritaire sont TOUJOURS gardées (elles n'entrent pas dans le quota).
QUOTA_TRAIN = {0: 800, 1: 800, 2: 800}
QUOTA_VAL   = {0: 100, 1: 100, 2: 100}

for split, quota in [('train', QUOTA_TRAIN), ('val', QUOTA_VAL)]:
    # Séparer frames par dominant + savoir lesquelles contiennent une minorité.
    dominant = collections.defaultdict(list)
    always_keep = []
    for lbl in (BAK / 'labels' / split).glob('*.txt'):
        dom, classes = frame_stats(lbl)
        if dom is None: continue
        if MINORITY & classes:
            always_keep.append(lbl)   # frame précieuse → sauvée d'office
        else:
            dominant[dom].append(lbl)

    random.seed(42)
    kept = list(always_keep)
    for cls_id, frames in dominant.items():
        cap = quota.get(cls_id, 99999)
        random.shuffle(frames)
        kept.extend(frames[:cap])

    for lbl in kept:
        stem = lbl.stem
        img_src = BAK / 'images' / split / f'{stem}.jpg'
        img_dst = MERGED / 'images' / split / f'{stem}.jpg'
        lbl_dst = MERGED / 'labels' / split / f'{stem}.txt'
        if img_src.exists() and not img_dst.exists():
            real = img_src.resolve() if img_src.is_symlink() else img_src
            try: img_dst.symlink_to(real)
            except OSError: shutil.copy(real, img_dst)
        lbl_dst.write_text(lbl.read_text())

    print(f'{split}: always_keep={len(always_keep)}, dominant_after_quota={len(kept)-len(always_keep)}, total={len(kept)}')

(MERGED / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(MERGED), 'train': 'images/train', 'val': 'images/val', 'names': CLASS_NAMES
}, sort_keys=False))

counts = [0] * len(CLASS_NAMES)
for lbl in (MERGED / 'labels').rglob('*.txt'):
    for line in lbl.read_text().splitlines():
        try: counts[int(line.split()[0])] += 1
        except: pass

print(f'\n=== APRÈS RÉÉQUILIBRAGE ===')
print(f'Frames  : {sum(1 for _ in (MERGED/"images").rglob("*.jpg"))}')
print(f'Annots  : {sum(counts)}')
for name, n in zip(CLASS_NAMES, counts):
    bar = '█' * min(60, n // max(1, max(counts) // 60))
    print(f'  {name:10s} {n:>7d}  {bar}')

## 10ter — Run B : warm-start depuis Run A sur données rééquilibrées

Repart du `best.pt` du Run A (qui connaît déjà nos datasets) plutôt que de COCO. LR petit + pas de freeze → adapte l'attention aux classes minoritaires sans détruire ce qui a été appris.

In [ ]:
from ultralytics import YOLO
model = YOLO('/workspace/runs/boeuf_yolo26s_832_mixed_warm/weights/best.pt')
model.train(
    data='/workspace/merged/data.yaml',
    imgsz=832,
    epochs=25,               # court — le modèle connaît déjà les datasets
    batch=-1,
    patience=10,
    lr0=0.001,               # LR petit = conserve savoir Run A
    freeze=0,                # pas de freeze = on veut adapter à la nouvelle balance
    cos_lr=True,
    close_mosaic=5,
    device=0,
    project='/workspace/runs',
    name='boeuf_yolo26s_832_mixed_balanced',
    save=True,
)

## 11 — Validation + matrice de confusion

Point clé à vérifier : `walking` doit avoir sa propre ligne/colonne dans `confusion_matrix_normalized.png`, distincte de `standing`/`eating`.

In [ ]:
metrics = model.val(data='/workspace/merged/data.yaml', imgsz=832)
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'mAP50    : {metrics.box.map50:.4f}\n')
print('Per-class mAP50 :')
for name, ap in zip(CANONICAL, metrics.box.maps):
    print(f'  {name:10s} {ap:.3f}')
!ls /workspace/runs/boeuf_yolo26s_832_mixed/*.png

## 12 — Export CoreML pour l'app Swift

In [ ]:
best = '/workspace/runs/boeuf_yolo26s_832_mixed/weights/best.pt'
YOLO(best).export(format='coreml', nms=True, imgsz=832)
!ls -lh /workspace/runs/boeuf_yolo26s_832_mixed/weights/best.mlpackage

## 13 — Tarball pour download

In [ ]:
import shutil
shutil.make_archive(
    '/tmp/boeuf_yolo26s_832_mixed', 'gztar',
    root_dir='/workspace/runs/boeuf_yolo26s_832_mixed/weights',
    base_dir='best.mlpackage',
)
print('✓ Prêt : /tmp/boeuf_yolo26s_832_mixed.tar.gz')
!ls -lh /tmp/boeuf_yolo26s_832_mixed.tar.gz